In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")

In [4]:
X = X_train
y = y_train
n_samples, n_features = X.shape

print(n_samples, n_features)
X.shape

8000 17


(8000, 17)

# **My own model**

In [52]:
class LinearRegressionManual():
    
    def __init__(self, learning_rate=0.1, epoch=1000, method="batch", batch_size=32, random_state=42):
        self.learning_rate = learning_rate
        self.epoch = epoch
        self.method = method
        self.batch_size = batch_size
        self.random_state = random_state
        self.cost_history = []
        self.weights = None
        self.bias = None

    def __repr__(self):
        return (f"LinearRegressionScratch(learning_rate={self.learning_rate}, "
                f"n_iterations={self.epoch})")

    def fit(self, X, y):

        X = np.array(X)
        y = np.array(y).ravel()
        
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for i in range(self.epoch):
            Y = X @ self.weights.T + self.bias
            error = y - Y
            cost = np.mean((error)**2)
            self.cost_history.append(cost)
    
            dw = (-2/n_samples) * (X.T @ error)
            db = (-2/n_samples) * (np.sum(error))

            self.weights -= self.learning_rate*dw
            self.bias -= self.learning_rate*db

            ss_res = np.sum((error)**2)
            ss_tot = np.sum((y - np.mean(y))**2)
            r2 = 1 - (ss_res / ss_tot)

            if i%100==0:
                print(f"Epoch {i:4d}: Loss={cost:.4f}, Score={r2:.4f}")

    def predict(self, X):
        X = np.array(X)
        return X @ self.weights + self.bias
    
    def learning_curve(self):
        plt.plot(self.cost_history, label="MSE Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Value")
        plt.title("Training Progress")
        plt.show()

    def metrics(self, y_pred, y_test):
        if(y_pred.shape!=y_test.shape):
            y_pred = np.array(y_pred).reshape(-1)
            y_test = np.array(y_test).reshape(-1)
            
        MSE = np.mean((y_test-y_pred)**2)
        RMSE = np.sqrt(MSE)
        MAE = np.mean(np.abs(y_test-y_pred))

        ss_res = np.sum((y_test-y_pred)**2)
        ss_tot = np.sum((y_test-np.mean(y_test))**2)
        r2 = 1-(ss_res/ss_tot)

        print(f"MSE {MSE:.3f}, RMSE {RMSE:.3f}, MAE {MAE:.3f}, R_sqr {r2:3f}")
        return MSE, RMSE, MAE, r2


In [53]:
model = LinearRegressionManual()
model.fit(X_train, y_train)

Epoch    0: Loss=58934088.7029, Score=-6.4979
Epoch  100: Loss=971870.7097, Score=0.8764
Epoch  200: Loss=971853.9844, Score=0.8764
Epoch  300: Loss=971853.9241, Score=0.8764
Epoch  400: Loss=971853.9235, Score=0.8764
Epoch  500: Loss=971853.9235, Score=0.8764
Epoch  600: Loss=971853.9235, Score=0.8764
Epoch  700: Loss=971853.9235, Score=0.8764
Epoch  800: Loss=971853.9235, Score=0.8764
Epoch  900: Loss=971853.9235, Score=0.8764


**Final Accuracy**

In [54]:
y_pred = model.predict(X_test)
MSE_, RMSE_, MAE_, R2_ = model.metrics(y_pred, y_test)


MSE 992545.383, RMSE 996.266, MAE 799.277, R_sqr 0.869430


In [21]:
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_train, y_train, cv=cv, scoring='r2')

print("CV R² Scores:", cv_scores)
print("Mean R²:", np.mean(cv_scores))


CV R² Scores: [0.86114105 0.88356287 0.86754645 0.88867743 0.87874017 0.87109857
 0.87951705 0.87036493 0.8809425  0.87292178]
Mean R²: 0.875451279924374


# **Lasso regression with gridSearch**

In [39]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso

param_grid = {'alpha': np.logspace(-3, 1, 20)}  # 0.001 → 10 range

lasso = Lasso(max_iter=10000)
grid_search = GridSearchCV(lasso, param_grid, cv=5, scoring='r2', n_jobs=-1)

grid_search.fit(X_train, y_train)

print("Best Alpha:", grid_search.best_params_['alpha'])
print("Best CV R²:", grid_search.best_score_)

lasso_final = grid_search.best_estimator_
lasso_final.fit(X_train, y_train)

y_pred_final = lasso_final.predict(X_test)


Best Alpha: 3.792690190732246
Best CV R²: 0.8753725272642245


In [40]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred_final)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_final)
r2 = r2_score(y_test, y_pred_final)

print(f"Final Test RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")

Final Test RMSE: 995.91, MAE: 798.88, R²: 0.8695


# **Model Comparision**

In [66]:
print(f"Manual model: {MSE_:.3f}, Inbuilt model: {mse:.3f} | Difference: {(MSE_ - mse):.3f}")
print(f"Manual model: {RMSE_:.3f}, Inbuilt model: {rmse:.3f} | Difference: {(RMSE_ - rmse):.3f}")
print(f"Manual model: {MAE_:.3f}, Inbuilt model: {mae:.3f} | Difference: {(MAE_ - mae):.3f}")
print(f"Manual model: {R2_:.4f}, Inbuilt model: {r2:.4f} | Difference: {(R2_ - r2):.4f}")

Manual model: 992545.383, Inbuilt model: 991833.511 | Difference: 711.872
Manual model: 996.266, Inbuilt model: 995.908 | Difference: 0.357
Manual model: 799.277, Inbuilt model: 798.884 | Difference: 0.393
Manual model: 0.8694, Inbuilt model: 0.8695 | Difference: -0.0001


In [69]:
import pickle, json

with open ("../models/used_car_prediction.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved!")

Model saved!
